In [ ]:
import os
import sys
import numpy as np
import torch
import scipy.signal as signal
import h5py
# Acceso a los módulos del TFM
sys.path.insert(0, r"c:\repos\DroneDetectionRF")
from NoisyUAV.funciones.dsp_rf.detector_entropia import detectar_bursts, plot_muestra, print_diagnostico
from NoisyUAV.modelos.cvcnn import ComplexConv1DNet
import torch.nn.functional as F
from NoisyUAV.modelos.burst_cvcnn import BurstCVCNN
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import SNR_estimation
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

In [ ]:
# ======================================================================
# 1. PARÁMETROS DEL DATASET DRFF-R2 (.MAT)
# ======================================================================
# Archivo de prueba (DJI MAVIC 3C)
archivo_drff_r2 = r"C:\TFM_data\DRFF-R2\mavic3C_1_c1.mat"
FS_ORIGINAL = 100e6  # 100 MHz (obtenido del paper DRFF-R2)
FS_TARGET = 14e6     # 14 MHz (la frecuencia de NoisyUAV)
MS_A_LEER = 500  # Vamos a inspeccionar medio segundo
MUESTRAS_A_LEER = int((MS_A_LEER / 1000) * FS_ORIGINAL)
print("==================================================")
print(f"📡 LECTURA DE DRFF-R2 (MATLAB DICT)")
print(f"Extrayendo primeros {MS_A_LEER} ms → {MUESTRAS_A_LEER} muestras IQ")

In [ ]:
# ======================================================================
# 2. CARGA DEL ARCHIVO .MAT MEDIANTE H5PY (Lectura Parcial Diferida)
# ======================================================================
if not os.path.exists(archivo_drff_r2):
    raise FileNotFoundError(f"Archivo no encontrado en: {archivo_drff_r2}")
print("⏳ Abriendo conector HDF5 al archivo v7.3...")
with h5py.File(archivo_drff_r2, 'r') as f:
    # Ver variables disponibles
    print(f"📦 Variables en archivo: {list(f.keys())}")
    
    # IMPORTANTE: A diferencia de loadmat, h5py nos permite hacer slicing 
    # directamente sobre el disco duro sin saturar la RAM con los 700MB.
    
    # H5PY guarda los arrays de Matlab transpuetos, si fuera [1, N] será [N, 1]
    # Extraemos solo las muestras que pedimos (0 a MUESTRAS_A_LEER)
    
    # Extraemos I y Q y aplanamos al vuelo y forzamos a float32
    print(f"Buscando leer las primeras {MUESTRAS_A_LEER} muestras...")
    I_orig = f['RF0_I'][:MUESTRAS_A_LEER].flatten().astype(np.float32)
    Q_orig = f['RF0_Q'][:MUESTRAS_A_LEER].flatten().astype(np.float32)
print(f"✅ Canales desentrelazados HDF5. Shapes: I={I_orig.shape}, Q={Q_orig.shape}")
# AGC (Automatic Gain Control base)
max_val = max(np.max(np.abs(I_orig)), np.max(np.abs(Q_orig)))
I_orig = I_orig / max_val
Q_orig = Q_orig / max_val

In [ ]:
# ======================================================================
# 3. DOWNSAMPLING (de 100 MHz a 14 MHz)
# ======================================================================
print(f"⏳ Filtro Anti-Aliasing y Decimador (100MHz -> 14MHz)...")
I_resampled = signal.resample_poly(I_orig, up=7, down=50)
Q_resampled = signal.resample_poly(Q_orig, up=7, down=50)
iq_tensor = torch.tensor(np.array([I_resampled, Q_resampled]), dtype=torch.float32)
print(f"✅ Tensor Final construido: {iq_tensor.shape}")

In [ ]:
# Variables Físicas de la Tesis
FS = 14e6
NPERSEG = 2048
Z_THRESH = 4.0     
MIN_BURST_MS = 0.5
MERGE_GAP_MS = 0.75
MIN_Z_ABS = 4.0
BG_MULT = 4
MAX_BINS_FRAC = 1
SMOOTH_MS = 0.1
ADAPTIVE_WINDOW_MS = 10 

# FS = 14e6
# NPERSEG = 2048
# Z_THRESH = 1.0       # <--- SUPER RELAJADO PARA CAZAR EL FONDO DEL RUIDO
# MIN_BURST_MS = 0.25  # <--- Más corto permitido
# MERGE_GAP_MS = 0.5
# MIN_Z_ABS = 1.0      # <--- Dejamos pasar todo
# BG_MULT = 4
# MAX_BINS_FRAC = 0.25 # <--- Aquí dejamos todo, que decida el Teacher
# SMOOTH_MS = 0.2
# ADAPTIVE_WINDOW_MS = 15 

# Detección (Extracción en crudo, SIN UMBRAL DINÁMICO, para ver si el Oráculo sabe distinguir)
t_ms, H, H_smooth, umbral_v, nf_v, ns, n_active, bursts = detectar_bursts(
    iq_tensor, fs=FS, nperseg=NPERSEG, z_thresh=Z_THRESH,
    min_burst_ms=MIN_BURST_MS, merge_gap_ms=MERGE_GAP_MS,
    min_z_abs=MIN_Z_ABS, bg_mult=BG_MULT, max_bins_frac=MAX_BINS_FRAC,
    smooth_ms=SMOOTH_MS, adaptive_window_ms=ADAPTIVE_WINDOW_MS,
)
print_diagnostico(
    t_ms, nf_v, ns, umbral_v, n_active, bursts,
    nperseg=NPERSEG, fs=FS, z_thresh=Z_THRESH,
    bg_mult=BG_MULT, max_bins_frac=MAX_BINS_FRAC,
    min_burst_ms=MIN_BURST_MS, merge_gap_ms=MERGE_GAP_MS,
    target=f"Target Prueba RFUAV", index=0,
)
# Magia Visual de tu Proyecto
fig_2d = plot_muestra(
    iq_tensor, t_ms, H, H_smooth, umbral_v, nf_v, ns, n_active, bursts,
    fs=FS, nperseg=NPERSEG, z_thresh=Z_THRESH,
    bg_mult=BG_MULT, max_bins_frac=MAX_BINS_FRAC,
    adaptive_window_ms=ADAPTIVE_WINDOW_MS,
)
fig_2d.show()

# Inferencia modelo Burst_norm

In [ ]:
# ======================================================================
# 5. INFERENCIA DEL MODELO ZERO-SHOT
# ======================================================================
if len(bursts) > 0:
    print("\n🧠 Inferencia CV-CNN sobre picos detectados")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = ComplexConv1DNet(num_classes=2, pool_output_size=64, dropout=0.5)
    
    ruta_pesos = r"C:\TFM_data\NoisyUAV\stage2_norm\checkpoints\cvcnn_best_model.pth"
    model.load_state_dict(torch.load(ruta_pesos, map_location=device, weights_only=True))
    model.to(device)
    model.eval()
    
    probs_dron = []
    with torch.no_grad():
        for i, b in enumerate(bursts):
            idx_inicio = int(b['t0'] * 1e-3 * FS_TARGET)
            idx_fin = int(b['t1'] * 1e-3 * FS_TARGET)
            pulso = iq_tensor[:, idx_inicio:idx_fin]
            
            # Normalización del pulso interceptado
            pulso_norm = pulso / torch.max(torch.abs(pulso))
            input_ia = pulso_norm.unsqueeze(0).to(device) 
            
            outputs = model(input_ia)
            prob = F.softmax(outputs, dim=1)
            prob_dron = prob[0][1].item() * 100 
            probs_dron.append(prob_dron)
            
            indicador = "✅" if prob_dron > 50 else "❌"
            print(f"  [DrffR2-B{i+1:02d}] t={b['t0']:6.2f} ms | IA predice: {prob_dron:6.2f}% probabilidad de DRON {indicador}")
            
    print("\n----------------------------------------------")
    prob_max = max(probs_dron)
    print(f"Veredicto Agregado (P_max): {prob_max:5.1f}%")
else:
    print("\nEl detector no aisló ningún Burst en este marco temporal.")

# Inferencia modelo alumn_v1

In [ ]:
# IMPORTANTE: Cargamos el modelo TEACHER (Oráculo entrenado en SNR >= 0)
ruta_pesos = r"c:\repos\DroneDetectionRF\NoisyUAV\modelo_alumn_v1\checkpoints\alumn_model_best.pt"
ckpt = torch.load(ruta_pesos, map_location=device, weights_only=False)
model = BurstCVCNN().to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()

# Rescatamos las estadísticas físicas (mean/std) originales con las que se entrenó el Teacher
phys_mean = torch.tensor(ckpt['phys_mean'], dtype=torch.float32).to(device)
phys_std  = torch.tensor(ckpt['phys_std'],  dtype=torch.float32).to(device)
print(f"✅ Nuevo modelo ALUMN cargado con éxito en {device.type.upper()}")
print(f"   Época Óptima del guardado: {ckpt.get('epoch', 'N/A')}")
print(f"   Validation F1: {ckpt.get('val_f1', 0.0):.4f}")

In [ ]:
print("==================================================")
print("     VEREDICTO ALUMNO (Inferencia en vivo)        ")
print("==================================================")
drones_encontrados = 0
ruidos_encontrados = 0

if len(bursts) == 0:
    print("  ❌ No se detectaron ráfagas. La sala se considera VACÍA (RUIDO).")
else:
    # --- AJUSTE DE CLIPPING (Para ser idéntico al entrenamiento) ---
    global_nf      = float(np.clip(np.median(nf_v), 0, 15))
    global_ns_val  = float(np.clip(ns, 0, 5))
    global_H_mean  = float(np.clip(np.mean(H_smooth), 0, 15))
    global_p75_act = float(np.clip(np.percentile(n_active, 75), 0, 2048))
    
    with torch.no_grad():
        for i, b in enumerate(bursts):
            # 1. RECORTAR Y NORMALIZAR ONDA
            idx_inicio = int(b['t0'] * 1e-3 * FS)
            idx_fin = int(b['t1'] * 1e-3 * FS)
            if idx_inicio >= idx_fin: continue
            
            pulso = iq_tensor[:, idx_inicio:idx_fin]
            power = pulso.pow(2).mean().clamp(min=1e-12).sqrt()
            pulso_normalizado = pulso / power
            
            # PADDING ESTÁNDAR (131072 muestras = 9.4 ms)
            TARGET_LEN = 131072
            C, L = pulso_normalizado.shape
            if L < TARGET_LEN:
                pad = torch.zeros(C, TARGET_LEN - L, device=pulso_normalizado.device)
                pulso_padded = torch.cat([pulso_normalizado, pad], dim=1)
            else:
                pulso_padded = pulso_normalizado[:, :TARGET_LEN]
                
            input_ia = pulso_padded.unsqueeze(0).to(device) 
            
            # 2. CONSTRUIR EL PERFIL FÍSICO (8 Dimensiones en orden exacto)
            feat_array = np.array([
                np.clip(b['dur_ms'], 0, 75),
                np.clip(abs(b['z_peak']), 0, 30),
                np.clip(b['drop_b'], 0, 10),
                np.clip(b['n_act'], 0, 2048),
                global_nf, 
                global_ns_val, 
                global_H_mean, 
                global_p75_act
            ], dtype=np.float32)
            
            feat_t = torch.from_numpy(feat_array).to(device)
            # Normalización manual con stats del checkpoint
            feat_norm = torch.clamp((feat_t - phys_mean) / (phys_std + 1e-8), -5.0, 5.0).unsqueeze(0)
            
            # 3. CLASIFICACIÓN
            logit = model(input_ia, feat_norm)
            prob_dron = torch.sigmoid(logit).item() * 100 
            
            t_ms_inicio = b['t0']
            if prob_dron > 50.0:
                drones_encontrados += 1
                print(f"  [B{i+1:02d}] t={t_ms_inicio:6.2f} ms | 🧠 IA: DRON   [{prob_dron:6.2f}%] ✅")
            else:
                ruidos_encontrados += 1
                print(f"  [B{i+1:02d}] t={t_ms_inicio:6.2f} ms | 🧠 IA: RUIDO  [{prob_dron:6.2f}%] ❌")

print("-" * 50)
print(f"RESUMEN: {drones_encontrados} Drones | {ruidos_encontrados} Ruidos")